# 🏥 Insurance Claims Portfolio — Exploratory Data Analysis

**Author:** Prajoshna Aare  
**Domain:** Insurance Analytics  
**Dataset:** 10,004 insurance policies across 5 product lines  
**Tools:** Python · Pandas · Matplotlib · Seaborn

---

### 🎯 Business Objective
Analyze a synthetic insurance portfolio to uncover operational inefficiencies, profitability risks, and customer segmentation patterns — answering the key questions every insurance analyst is expected to address:

1. Where is the claim rejection problem worst?
2. Which product lines are financially sustainable?
3. Where is capital stuck in unresolved claims?
4. Which customer segments carry the most risk?
5. Are premiums adequately priced against payouts?
6. How much of their coverage are customers actually utilizing?

---
## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})
COLORS = ['#2196F3', '#F44336', '#FF9800', '#4CAF50', '#9C27B0']
STATUS_COLORS = {'Settled': '#4CAF50', 'Pending': '#FF9800', 'Rejected': '#F44336'}

# ── Load ───────────────────────────────────────────────
df = pd.read_csv('InsuranceData.csv')
df['PolicyStartDate'] = pd.to_datetime(df['PolicyStartDate'])
df['PolicyEndDate']   = pd.to_datetime(df['PolicyEndDate'])
df['ClaimDate']       = pd.to_datetime(df['ClaimDate'], errors='coerce')

# Derived columns
df['AgeGroup']         = pd.cut(df['Age'], bins=[17,30,45,60,100],
                                labels=['18–30','31–45','46–60','60+'])
df['UtilizationRate']  = (df['ClaimAmount'] / df['CoverageAmount']).round(4)
df['PolicyDuration']   = (df['PolicyEndDate'] - df['PolicyStartDate']).dt.days

print(f'Dataset shape : {df.shape}')
print(f'Policy types  : {df.PolicyType.unique()}')
print(f'Claim statuses: {df.ClaimStatus.unique()}')
df.head()

---
## 2. Dataset Overview & Data Quality Check

In [ ]:
print('=== SHAPE ===')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')

print('\n=== NULL VALUES ===')
print(df.isnull().sum())

print('\n=== DESCRIPTIVE STATS (Numeric) ===')
df[['Age','PremiumAmount','CoverageAmount','ClaimAmount']].describe().round(2)

In [ ]:
# Distribution of numeric features
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
cols  = ['Age', 'PremiumAmount', 'CoverageAmount', 'ClaimAmount']
titles = ['Age Distribution', 'Premium Amount', 'Coverage Amount', 'Claim Amount']

for ax, col, title in zip(axes, cols, titles):
    ax.hist(df[col], bins=30, color='#2196F3', edgecolor='white', alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

plt.suptitle('Distribution of Key Numeric Features', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('01_distributions.png', bbox_inches='tight')
plt.show()

---
## Analysis 1 — Claim Rejection & Settlement Rate
> **Business Question:** Where is the rejection problem worst — and which product line has the healthiest settlement rate?

**Why this matters:** Rejection rate is a direct KPI for customer trust. High rejection rates lead to policy cancellations, negative reviews, and regulatory scrutiny.

In [ ]:
# Overall claim status
status_counts = df['ClaimStatus'].value_counts()
status_pct    = (status_counts / len(df) * 100).round(1)

print('=== OVERALL CLAIM STATUS ===')
for s, n, p in zip(status_counts.index, status_counts.values, status_pct.values):
    print(f'  {s:10s}: {n:,}  ({p}%)')

# By policy type
ct = pd.crosstab(df['PolicyType'], df['ClaimStatus'], normalize='index').round(3) * 100
ct = ct[['Settled','Pending','Rejected']]
print('\n=== CLAIM STATUS % BY POLICY TYPE ===')
print(ct.round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Donut chart — overall
ax = axes[0]
colors = [STATUS_COLORS[s] for s in status_counts.index]
wedges, texts, autotexts = ax.pie(
    status_counts, labels=status_counts.index, autopct='%1.1f%%',
    colors=colors, startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2)
)
for at in autotexts:
    at.set_fontsize(11); at.set_fontweight('bold')
ax.set_title('Overall Claim Status Distribution\n(10,004 Policies)', pad=15)

# Stacked bar — by policy type
ax2 = axes[1]
ct_plot = ct[['Settled','Pending','Rejected']]
bottom = np.zeros(len(ct_plot))
bar_colors = [STATUS_COLORS['Settled'], STATUS_COLORS['Pending'], STATUS_COLORS['Rejected']]
for col, color in zip(ct_plot.columns, bar_colors):
    bars = ax2.bar(ct_plot.index, ct_plot[col], bottom=bottom, label=col,
                   color=color, edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, ct_plot[col]):
        if val > 5:
            ax2.text(bar.get_x() + bar.get_width()/2,
                     bar.get_y() + bar.get_height()/2,
                     f'{val:.0f}%', ha='center', va='center',
                     fontsize=9, fontweight='bold', color='white')
    bottom += ct_plot[col].values

ax2.set_title('Claim Status % by Policy Type')
ax2.set_ylabel('Percentage (%)')
ax2.set_ylim(0, 105)
ax2.legend(loc='upper right')

plt.suptitle('Analysis 1: Claim Rejection & Settlement Rate', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('02_claim_status.png', bbox_inches='tight')
plt.show()

print('\n💡 INSIGHT: 43.5% of all claims are rejected. Life Insurance has the worst settlement rate at 30.9%.')

---
## Analysis 2 — Loss Ratio by Policy Type
> **Business Question:** Which product lines are paying out more than they earn in premiums?

**Why this matters:** Loss Ratio (Settled Claims ÷ Premium Collected) is the #1 profitability metric in insurance. A ratio above 1.0x means the insurer loses money on that product.

In [ ]:
settled_df = df[df['ClaimStatus'] == 'Settled']

loss_ratio = (
    settled_df.groupby('PolicyType')['ClaimAmount'].sum() /
    df.groupby('PolicyType')['PremiumAmount'].sum()
).round(3).sort_values(ascending=False)

print('=== LOSS RATIO BY POLICY TYPE ===')
for ptype, ratio in loss_ratio.items():
    flag = '🔴 LOSS' if ratio > 1 else '🟢 OK'
    print(f'  {ptype:10s}: {ratio:.2f}x  {flag}')

print(f'\n  Industry benchmark: 0.60–0.80x')
print(f'  All product lines are above 1.0x → premiums are severely underpriced')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

bar_colors = ['#F44336' if r > 1 else '#4CAF50' for r in loss_ratio.values]
bars = ax.barh(loss_ratio.index, loss_ratio.values, color=bar_colors,
               edgecolor='white', height=0.55)

for bar, val in zip(bars, loss_ratio.values):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}x', va='center', fontsize=11, fontweight='bold')

ax.axvline(x=1.0, color='black', linestyle='--', linewidth=1.5, label='Break-even (1.0x)')
ax.axvline(x=0.75, color='grey', linestyle=':', linewidth=1.2, label='Industry target (0.75x)')
ax.set_xlim(0, 2.1)
ax.set_xlabel('Loss Ratio (Settled Claims ÷ Premium Collected)')
ax.set_title('Analysis 2: Loss Ratio by Policy Type\n(All lines above break-even — premiums underpriced)', pad=12)
ax.legend()

plt.tight_layout()
plt.savefig('03_loss_ratio.png', bbox_inches='tight')
plt.show()

print('\n💡 INSIGHT: Auto insurance has the highest loss ratio at 1.74x — for every ₹1 earned, ₹1.74 is paid out.')

---
## Analysis 3 — High-Value Claims & Fraud Risk
> **Business Question:** Which product lines are concentrated with high-value claims — a potential fraud or mispricing red flag?

**Why this matters:** Claims significantly above average signal either fraud, mis-selling, or policy gaps. Identifying concentration helps prioritize investigations.

In [ ]:
threshold = df['ClaimAmount'].quantile(0.75)   # top 25%
high_df   = df[df['ClaimAmount'] > threshold]

print(f'High-value claim threshold (75th percentile): ₹{threshold:,.0f}')
print(f'Total high-value claims: {len(high_df):,} ({len(high_df)/len(df)*100:.1f}% of portfolio)\n')

hv_dist = high_df['PolicyType'].value_counts()
hv_pct  = (hv_dist / hv_dist.sum() * 100).round(1)
print('=== HIGH-VALUE CLAIM CONCENTRATION ===')
for pt, n, p in zip(hv_dist.index, hv_dist.values, hv_pct.values):
    print(f'  {pt:10s}: {n:,} claims  ({p}%)')

print('\n=== AVG CLAIM AMOUNT (Settled) BY POLICY TYPE ===')
print(settled_df.groupby('PolicyType')['ClaimAmount'].mean().sort_values(ascending=False).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Pie — high value claim share
ax1 = axes[0]
ax1.pie(hv_dist.values, labels=hv_dist.index, autopct='%1.1f%%',
        colors=COLORS, startangle=90,
        wedgeprops=dict(width=0.6, edgecolor='white', linewidth=2))
ax1.set_title(f'High-Value Claims (>₹{threshold:,.0f}) by Policy Type')

# Scatter — premium vs claim amount (settled)
ax2 = axes[1]
pt_colors = {'Auto':'#2196F3','Health':'#F44336','Home':'#FF9800',
             'Life':'#4CAF50','Travel':'#9C27B0'}
for ptype, grp in settled_df.groupby('PolicyType'):
    ax2.scatter(grp['PremiumAmount'], grp['ClaimAmount'],
                alpha=0.25, s=15, label=ptype, color=pt_colors[ptype])

ax2.set_xlabel('Premium Amount (₹)')
ax2.set_ylabel('Claim Amount (₹)')
ax2.set_title('Premium Paid vs Claim Amount (Settled Claims)')
ax2.legend(markerscale=2)

plt.suptitle('Analysis 3: High-Value Claims & Fraud Risk Concentration', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('04_high_value_claims.png', bbox_inches='tight')
plt.show()

print('\n💡 INSIGHT: Travel Insurance accounts for 40.6% of all high-value claims despite being 1 of 5 product lines.')

---
## Analysis 4 — Customer Risk Segmentation by Age Group
> **Business Question:** Which age segments have the highest rejection rates — and which are most profitable to insure?

**Why this matters:** Age is the primary underwriting variable. Identifying which segments are underserved or over-rejected helps in product design and pricing strategy.

In [ ]:
age_ct = pd.crosstab(df['AgeGroup'], df['ClaimStatus'], normalize='index').round(3) * 100
age_ct = age_ct[['Settled','Pending','Rejected']]

print('=== CLAIM STATUS % BY AGE GROUP ===')
print(age_ct.round(1))

print('\n=== POLICY COUNT BY AGE GROUP ===')
print(df['AgeGroup'].value_counts().sort_index())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Grouped bar — rejection & settlement by age
ax1 = axes[0]
x    = np.arange(len(age_ct))
w    = 0.35
ax1.bar(x - w/2, age_ct['Settled'],  width=w, label='Settled',  color='#4CAF50', edgecolor='white')
ax1.bar(x + w/2, age_ct['Rejected'], width=w, label='Rejected', color='#F44336', edgecolor='white')
ax1.set_xticks(x)
ax1.set_xticklabels(age_ct.index)
ax1.set_ylabel('Percentage (%)')
ax1.set_title('Settlement vs Rejection Rate by Age Group')
ax1.legend()
ax1.set_ylim(0, 60)

for i, (s, r) in enumerate(zip(age_ct['Settled'], age_ct['Rejected'])):
    ax1.text(i - w/2, s + 0.8, f'{s:.1f}%', ha='center', fontsize=9, fontweight='bold', color='#2e7d32')
    ax1.text(i + w/2, r + 0.8, f'{r:.1f}%', ha='center', fontsize=9, fontweight='bold', color='#c62828')

# Box — claim amount distribution by age
ax2 = axes[1]
age_order = ['18–30','31–45','46–60','60+']
settled_age = settled_df[settled_df['AgeGroup'].notna()]
groups = [settled_age[settled_age['AgeGroup']==ag]['ClaimAmount'].values for ag in age_order]
bp = ax2.boxplot(groups, labels=age_order, patch_artist=True,
                 medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], ['#2196F3','#FF9800','#4CAF50','#9C27B0']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax2.set_ylabel('Settled Claim Amount (₹)')
ax2.set_title('Settled Claim Amount Distribution by Age Group')

plt.suptitle('Analysis 4: Customer Risk Segmentation by Age Group', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('05_age_segmentation.png', bbox_inches='tight')
plt.show()

print('\n💡 INSIGHT: 18–30 age group has the highest rejection rate at 45.8% — a major customer retention risk.')

---
## Analysis 5 — Premium Adequacy Check
> **Business Question:** Are premiums in line with what is actually being claimed, or are certain products severely underpriced?

**Why this matters:** If average claim amount consistently exceeds average premium, the product is underpriced — a direct threat to solvency.

In [ ]:
premium_avg = df.groupby('PolicyType')['PremiumAmount'].mean().round(2)
claim_avg   = settled_df.groupby('PolicyType')['ClaimAmount'].mean().round(2)

adequacy = pd.DataFrame({'Avg Premium': premium_avg, 'Avg Settled Claim': claim_avg})
adequacy['Gap (Claim - Premium)'] = (adequacy['Avg Settled Claim'] - adequacy['Avg Premium']).round(2)
adequacy['Underpriced?'] = adequacy['Gap (Claim - Premium)'].apply(lambda x: '🔴 YES' if x > 0 else '🟢 NO')

print('=== PREMIUM ADEQUACY BY POLICY TYPE ===')
print(adequacy)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

x   = np.arange(len(adequacy))
w   = 0.38
b1  = ax.bar(x - w/2, adequacy['Avg Premium'], width=w, label='Avg Premium',
             color='#2196F3', edgecolor='white')
b2  = ax.bar(x + w/2, adequacy['Avg Settled Claim'], width=w, label='Avg Settled Claim',
             color='#F44336', edgecolor='white')

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'₹{bar.get_height():,.0f}', ha='center', fontsize=9, color='#1565C0')
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'₹{bar.get_height():,.0f}', ha='center', fontsize=9, color='#b71c1c')

ax.set_xticks(x)
ax.set_xticklabels(adequacy.index)
ax.set_ylabel('Amount (₹)')
ax.set_ylim(0, adequacy['Avg Settled Claim'].max() * 1.18)
ax.set_title('Analysis 5: Average Premium vs Average Settled Claim Amount\n(All product lines show claims exceeding premiums)', pad=12)
ax.legend()

plt.tight_layout()
plt.savefig('06_premium_adequacy.png', bbox_inches='tight')
plt.show()

print('\n💡 INSIGHT: Life Insurance has the widest gap — avg settled claim is ₹3,077 vs avg premium of ₹589. Severely underpriced.')

---
## Analysis 6 — Pending Claims Financial Exposure
> **Business Question:** Where is unresolved capital concentrated — and which policy types have the most backlog?

**Why this matters:** Pending claims tie up loss reserves, inflate operational costs, and signal process inefficiencies in claim handling.

In [ ]:
pending_df = df[df['ClaimStatus'] == 'Pending']

exposure = pending_df.groupby('PolicyType').agg(
    PendingCount=('ClaimAmount','count'),
    TotalExposure=('ClaimAmount','sum'),
    AvgClaim=('ClaimAmount','mean')
).round(2).sort_values('TotalExposure', ascending=False)

print(f'=== PENDING CLAIMS FINANCIAL EXPOSURE ===')
print(f'Total pending claims  : {len(pending_df):,}')
print(f'Total capital at risk : ₹{pending_df["ClaimAmount"].sum():,.2f}\n')
print(exposure)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Total exposure bar
ax1 = axes[0]
bars = ax1.bar(exposure.index, exposure['TotalExposure'], color=COLORS, edgecolor='white')
for bar in bars:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
             f'₹{bar.get_height()/1e5:.2f}L', ha='center', fontsize=10, fontweight='bold')
ax1.set_ylabel('Total Pending Exposure (₹)')
ax1.set_title('Total Financial Exposure by Policy Type\n(Pending Claims Only)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e5:.1f}L'))

# Pending count by age group
ax2 = axes[1]
age_pending = pending_df['AgeGroup'].value_counts().sort_index()
ax2.bar(age_pending.index.astype(str), age_pending.values,
        color=['#2196F3','#FF9800','#4CAF50','#9C27B0'], edgecolor='white')
for i, v in enumerate(age_pending.values):
    ax2.text(i, v + 5, str(v), ha='center', fontsize=10, fontweight='bold')
ax2.set_ylabel('Number of Pending Claims')
ax2.set_title('Pending Claims Count by Age Group')

plt.suptitle('Analysis 6: Pending Claims Financial Exposure', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('07_pending_exposure.png', bbox_inches='tight')
plt.show()

print('\n💡 INSIGHT: ₹68.07L is locked in pending claims. Travel has the highest exposure among all policy types.')

---
## Analysis 7 — Coverage Utilization Rate
> **Business Question:** How much of their coverage are customers actually claiming — and are any segments over/under-insured?

**Why this matters:** Low utilization = over-insured customers (upsell opportunity). High utilization = underpriced coverage (risk alert). This drives renewal and pricing strategy.

In [ ]:
util_by_type = settled_df.groupby('PolicyType')['UtilizationRate'].agg(['mean','median','max']).round(4)
util_by_type.columns = ['Avg Utilization','Median Utilization','Max Utilization']

print('=== COVERAGE UTILIZATION RATE (Settled Claims Only) ===')
print(util_by_type)
print('\nNote: 1.0 = customer claimed 100% of their coverage amount')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram — utilization rate overall
ax1 = axes[0]
util_data = settled_df['UtilizationRate'].replace([np.inf, -np.inf], np.nan).dropna()
util_data = util_data[util_data <= 1.5]
ax1.hist(util_data, bins=40, color='#2196F3', edgecolor='white', alpha=0.85)
ax1.axvline(util_data.mean(), color='red', linestyle='--', linewidth=2,
            label=f'Mean: {util_data.mean():.2f}')
ax1.axvline(0.5, color='orange', linestyle=':', linewidth=1.5, label='50% utilization')
ax1.set_xlabel('Utilization Rate (Claim ÷ Coverage)')
ax1.set_ylabel('Count')
ax1.set_title('Coverage Utilization Rate Distribution\n(Settled Claims)')
ax1.legend()

# Avg utilization by policy type
ax2 = axes[1]
util_pt = util_by_type['Avg Utilization'].sort_values(ascending=False)
bars = ax2.bar(util_pt.index, util_pt.values, color=COLORS, edgecolor='white')
ax2.axhline(y=0.5, color='red', linestyle='--', linewidth=1.5, label='50% benchmark')
for bar in bars:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{bar.get_height():.2f}', ha='center', fontsize=10, fontweight='bold')
ax2.set_ylabel('Avg Utilization Rate')
ax2.set_title('Average Coverage Utilization Rate by Policy Type')
ax2.legend()

plt.suptitle('Analysis 7: Coverage Utilization Rate', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('08_utilization.png', bbox_inches='tight')
plt.show()

print('\n💡 INSIGHT: Most customers utilize <50% of their coverage — signaling over-insurance and a premium repricing opportunity.')

---
## Summary Dashboard — Key Business Insights

| # | Insight | Metric | Business Action |
|---|---------|--------|----------------|
| 1 | High claim rejection rate | 43.5% rejected | Review underwriting criteria & claim communication |
| 2 | All product lines loss-making | Loss ratio 1.61x–1.74x | Reprice premiums across all lines |
| 3 | Travel = high-value claim hotspot | 40.6% of high-value claims | Tighten Travel sub-limits & add exclusions |
| 4 | Young adults face highest rejection | 45.8% rejection (18–30) | Improve policy onboarding & claim guidance |
| 5 | Life insurance worst settlement rate | 30.9% settlement | Review Life claim eligibility criteria |
| 6 | ₹68L frozen in pending claims | 2,263 pending cases | Fast-track claim resolution workflows |
| 7 | Customers under-utilizing coverage | Avg ~50% utilization | Target renewal upsell or right-size coverage |

In [ ]:
print('✅ EDA Complete')
print('All charts saved as PNG files in the working directory.')
print('\nFiles generated:')
import os
for f in sorted(os.listdir('.')):
    if f.endswith('.png'):
        print(f'  📊 {f}')